In [1]:
# Cell 1 — imports and load all 10 payloads
#
# Run capture_exemplar_payloads.py first if the JSON files don't exist.
# output/edop/surface/exemplars/ is gitignored; files live on disk only.

import sys
import json
from pathlib import Path
from collections import Counter

import pandas as pd

sys.path.insert(0, '../../../..')
import scripts.shared.db_utils as _dbu

ROOT = Path(_dbu.__file__).parent.parent.parent
OUT  = ROOT / 'output' / 'edop' / 'surface' / 'exemplars'

assert OUT.exists(), f'Run capture_exemplar_payloads.py first — {OUT} not found'

def load(name):
    return json.loads((OUT / name).read_text())

p = {
    's1_lean':   load('01_single_basin_lean.json'),
    's1_detail': load('01_single_basin_detail.json'),
    's2_lean':   load('02_buffer_lean.json'),
    's2_detail': load('02_buffer_detail.json'),
    's3_lean':   load('03_polity_nsong_lean.json'),
    's3_detail': load('03_polity_nsong_detail.json'),
    's4_lean':   load('04_basin_ring_lean.json'),
    's4_detail': load('04_basin_ring_detail.json'),
    's5_lean':   load('05_polygon_4corners_lean.json'),
    's5_detail': load('05_polygon_4corners_detail.json'),
}

print('Loaded 10 payloads.')
for k, v in p.items():
    rows = v.get('rows') or []
    print(f'  {k:<15s}  rows={len(rows)}')

Loaded 10 payloads.
  s1_lean          rows=52
  s1_detail        rows=52
  s2_lean          rows=52
  s2_detail        rows=52
  s3_lean          rows=372
  s3_detail        rows=372
  s4_lean          rows=0
  s4_detail        rows=0
  s5_lean          rows=52
  s5_detail        rows=52


In [2]:
# Cell 2 — top-level payload keys and neighborhood block per scope
#
# Confirms payload envelope shape and what each scope puts in 'neighborhood'.
# Basin-ring is the structural outlier — no top-level 'rows'.

scopes = [
    ('S1 single-basin', 's1_lean'),
    ('S2 buffer',       's2_lean'),
    ('S3 polity',       's3_lean'),
    ('S4 basin-ring',   's4_lean'),
    ('S5 4-corners',    's5_lean'),
]

for label, key in scopes:
    payload = p[key]
    print(f'--- {label} ---')
    print(f'  top-level keys:  {sorted(payload.keys())}')
    nbhd = payload.get('neighborhood') or payload.get('center', {}).get('neighborhood')
    print(f'  neighborhood:    {nbhd}')
    print(f'  shortfall:       {payload.get("shortfall", "N/A (ring)")}')    
    print(f'  temporal:        {payload.get("temporal")}')
    print(f'  caveats:         {payload.get("caveats")}')
    print()

--- S1 single-basin ---
  top-level keys:  ['bands', 'caveats', 'neighborhood', 'rows', 'shortfall', 'temporal']
  neighborhood:    {'type': 'basin', 'lat': 16.8167, 'lon': -2.9833, 'level': 6, 'hybas_id': 1060551560, 'n_units': 1, 'unit_type': 'basin'}
  shortfall:       0.0
  temporal:        None
  caveats:         {}

--- S2 buffer ---
  top-level keys:  ['bands', 'caveats', 'neighborhood', 'rows', 'shortfall', 'temporal']
  neighborhood:    {'type': 'buffer', 'lat': 16.8167, 'lon': -2.9833, 'radius_km': 100.0, 'level': 6, 'n_units': 9, 'unit_type': 'basin'}
  shortfall:       0.0
  temporal:        None
  caveats:         {}

--- S3 polity ---
  top-level keys:  ['bands', 'caveats', 'modality_post_pass', 'neighborhood', 'rows', 'shortfall', 'temporal']
  neighborhood:    {'type': 'polygon', 'level': 6, 'n_units': 376, 'unit_type': 'basin', 'marginal_exposure': {'lt_50pct': 0.029522, 'lt_20pct': 0.008443}}
  shortfall:       0.010546
  temporal:        {'from_year': 1000, 'to_year'

In [3]:
# Cell 3 — methods inventory (checklist item 1)
#
# Which method values appear, and how many rows each, across all scopes.
# Basin-ring shows center + one ring member separately.
# S3 polity includes Band T rows — separated out at the end.

print('=== BASE ROWS (Bands A-E) ===')
for label, key in [
    ('S1 single-basin', 's1_lean'),
    ('S2 buffer',       's2_lean'),
    ('S3 polity (base only)', 's3_lean'),
    ('S5 4-corners',    's5_lean'),
]:
    rows = [r for r in p[key]['rows'] if r.get('band') != 'T']
    c = Counter(r['method'] for r in rows)
    print(f'  {label}: {dict(c)}')

# basin-ring
ring = p['s4_lean']
c_center = Counter(r['method'] for r in ring['center']['rows'])
c_member = Counter(r['method'] for r in ring['ring'][0]['signature']['rows'])
print(f'  S4 ring center:   {dict(c_center)}')
print(f'  S4 ring member 0: {dict(c_member)}')

print()
print('=== BAND T ROWS (S3 polity only) ===')
t_rows = [r for r in p['s3_lean']['rows'] if r.get('band') == 'T']
print(f'  Total Band T rows: {len(t_rows)}')
print(f'  By method:    {dict(Counter(r["method"]    for r in t_rows))}')
print(f'  By unit_type: {dict(Counter(r["unit_type"] for r in t_rows))}')
print(f'  By variable:  {dict(Counter(r["variable"]  for r in t_rows))}')

=== BASE ROWS (Bands A-E) ===
  S1 single-basin: {'area_weighted': 34, 'dominant_basin': 3, 'class_mixture': 10, 'flag_fraction': 1, 'distribution_only': 3, 'extreme': 1}
  S2 buffer: {'area_weighted': 34, 'dominant_basin': 3, 'class_mixture': 10, 'flag_fraction': 1, 'distribution_only': 3, 'extreme': 1}
  S3 polity (base only): {'area_weighted': 34, 'dominant_basin': 3, 'class_mixture': 10, 'flag_fraction': 1, 'distribution_only': 3, 'extreme': 1}
  S5 4-corners: {'area_weighted': 34, 'dominant_basin': 3, 'class_mixture': 10, 'flag_fraction': 1, 'distribution_only': 3, 'extreme': 1}
  S4 ring center:   {'area_weighted': 34, 'dominant_basin': 3, 'class_mixture': 10, 'flag_fraction': 1, 'distribution_only': 3, 'extreme': 1}
  S4 ring member 0: {'area_weighted': 34, 'dominant_basin': 3, 'class_mixture': 10, 'flag_fraction': 1, 'distribution_only': 3, 'extreme': 1}

=== BAND T ROWS (S3 polity only) ===
  Total Band T rows: 320
  By method:    {'grid_areal_distribution': 311, 'global_forci

In [ ]:
# Cell 4 — histogram presence (checklist item 2)
#
# Which methods carry a distribution object in detail mode?
# 'detail' sub-dict is null in lean; present in detail.
# 'distribution' within detail is the histogram object.

print('=== HISTOGRAM PRESENCE IN DETAIL MODE ===')
for label, key in [
    ('S1 single-basin', 's1_detail'),
    ('S2 buffer',       's2_detail'),
    ('S3 polity (base)', 's3_detail'),
    ('S5 4-corners',    's5_detail'),
]:
    base = [r for r in p[key]['rows'] if r.get('band') != 'T']
    with_hist    = [r for r in base if r.get('detail') and r['detail'].get('distribution')]
    without_hist = [r for r in base if r.get('detail') and not r['detail'].get('distribution')]
    no_detail    = [r for r in base if not r.get('detail')]
    print(f'  {label}:')
    print(f'    histogram present: {len(with_hist)}  '
          f'(methods: {dict(Counter(r["method"] for r in with_hist))})')
    print(f'    detail but no hist: {len(without_hist)}  '
          f'(methods: {dict(Counter(r["method"] for r in without_hist))})')

# Band T histograms in S3
print()
t_detail = [r for r in p['s3_detail']['rows'] if r.get('band') == 'T']
t_hist   = [r for r in t_detail if r.get('detail') and r['detail'].get('distribution')]
t_no_hist = [r for r in t_detail if not (r.get('detail') and r['detail'].get('distribution'))]
print(f'  S3 Band T rows with histogram:    {len(t_hist)}  '
      f'(methods: {dict(Counter(r["method"] for r in t_hist))})')
print(f'  S3 Band T rows without histogram: {len(t_no_hist)}  '
      f'(methods: {dict(Counter(r["method"] for r in t_no_hist))})')

In [ ]:
# Cell 5 — histogram object anatomy
#
# Show one complete histogram object from B1 (area_weighted)
# and one from Band T (LMR grid_areal_distribution).
# These are the two contexts where histograms appear.

print('=== B1 HISTOGRAM (S2 buffer, first area_weighted row) ===')
b1 = next(r for r in p['s2_detail']['rows'] if r['method'] == 'area_weighted')
h  = b1['detail']['distribution']
print(f'  variable:    {b1["variable"]}')
print(f'  score:       {b1["representative_score"]}   coherence: {b1.get("coherence")}')
print(f'  hist keys:   {sorted(h.keys())}')
print(f'  bins:        {len(h["bins"])} edges')
print(f'  weights:     {len(h["weights"])} bars')
print(f'  unit_type:   {h["unit_type"]}   n_units: {h["n_units"]}   low_resolution: {h["low_resolution"]}')
print(f'  range:       min={h["min"]:.2f}  max={h["max"]:.2f}  p10={h["p10"]:.2f}  p90={h["p90"]:.2f}  mean={h["mean"]:.2f}')
print(f'  stamps:      resolver_year={h["resolver_year"]}  band_t_from={h["band_t_from"]}  band_t_to={h["band_t_to"]}')

print()
print('=== LMR HISTOGRAM (S3 polity, lmr_pdsi year=1000) ===')
lmr = next(r for r in p['s3_detail']['rows']
           if r['variable'] == 'lmr_pdsi' and r.get('year') == 1000)
h2  = lmr['detail']['distribution']
print(f'  variable:    {lmr["variable"]}   year: {lmr["year"]}')
print(f'  score:       {lmr["representative_score"]}  (null by design — distribution only)')
print(f'  hist keys:   {sorted(h2.keys())}')
print(f'  unit_type:   {h2["unit_type"]}   n_units: {h2["n_units"]}')
print(f'  range:       min={h2["min"]:.4f}  max={h2["max"]:.4f}  mean={h2["mean"]:.4f}')
print(f'  stamps:      resolver_year={h2["resolver_year"]}  band_t_from={h2["band_t_from"]}  band_t_to={h2["band_t_to"]}')

In [ ]:
# Cell 6 — Band T: LMR time-series structure (checklist item 3)
#
# One row per year per LMR variable. Show how the time series is reconstructed
# by collating rows: filter by variable, sort by year, read distribution.mean.
# This is what the envelope chart in the UI must do client-side.

t_rows = [r for r in p['s3_detail']['rows'] if r.get('band') == 'T']
lmr_rows = [r for r in t_rows if r['unit_type'] == 'lmr_cell']

print(f'Total LMR rows: {len(lmr_rows)}')
print(f'Variables: {sorted(set(r["variable"] for r in lmr_rows))}')
print(f'Year range: {min(r["year"] for r in lmr_rows)} – {max(r["year"] for r in lmr_rows)}')
print()

# Reconstruct lmr_pdsi as a time series
pdsi = sorted([r for r in lmr_rows if r['variable'] == 'lmr_pdsi'], key=lambda r: r['year'])
print('lmr_pdsi time series (first 10 years, using distribution.mean):')
print(f'  {"year":>6}  {"mean":>8}  {"p10":>8}  {"p90":>8}  {"score":>8}')
for r in pdsi[:10]:
    d = r['detail']['distribution']
    print(f'  {r["year"]:>6}  {d["mean"]:>8.4f}  {d["p10"]:>8.4f}  {d["p90"]:>8.4f}  {str(r["representative_score"]):>8}')

print()
print('Note: representative_score is null for all LMR rows — distribution.mean is the time-series value.')
print('Time-series collation requires detail mode.')

In [ ]:
# Cell 7 — Band T: HYDE and eVolv2k rows (checklist item 3 cont.)
#
# HYDE uses epoch_year (not year), reports raw km² values, no score.
# eVolv2k uses year, reports vssi in representative_raw, no score.
# Neither carries a histogram.

t_rows = [r for r in p['s3_lean']['rows'] if r.get('band') == 'T']

print('=== HYDE ROWS ===')
hyde = [r for r in t_rows if r['unit_type'] == 'hyde_cell']
print(f'  Count: {len(hyde)}')
print(f'  {"variable":<20}  {"epoch_year":>10}  {"raw (km²)":>12}  {"score":>8}')
for r in hyde:
    print(f'  {r["variable"]:<20}  {str(r["epoch_year"]):>10}  '
          f'{r["representative_raw"]:>12.4f}  {str(r["representative_score"]):>8}')

print()
print('=== eVolv2k ROWS ===')
evolv = [r for r in t_rows if r['method'] == 'global_forcing']
print(f'  Count: {len(evolv)} events in the query span')
print(f'  {"year":>6}  {"vssi (Tg SO₂)":>14}  {"score":>8}')
for r in sorted(evolv, key=lambda r: r['year']):
    print(f'  {r["year"]:>6}  {r["representative_raw"]:>14.2f}  {str(r["representative_score"]):>8}')

In [ ]:
# Cell 8 — temporal axes location (checklist item 4)
#
# Two independent axes:
#   resolver_year: which polity boundary year was used
#   from_year/to_year: Band T aggregation window
#
# Where do they each appear in the payload?

payload = p['s3_detail']

print('=== WHERE THE TWO TEMPORAL AXES LIVE (S3 polity) ===')
print()
print('payload["temporal"]:', payload.get('temporal'))
print('payload["neighborhood"] keys:', sorted(payload['neighborhood'].keys()))
print('  → resolver_year in neighborhood?', 'resolver_year' in payload['neighborhood'])

# Check histogram stamps
base_rows = [r for r in payload['rows'] if r.get('band') != 'T']
b1_with_hist = next(r for r in base_rows
                    if r['method'] == 'area_weighted' and r.get('detail', {}).get('distribution'))
h_b1 = b1_with_hist['detail']['distribution']
print()
print(f'B1 histogram stamp (variable={b1_with_hist["variable"]}):')
print(f'  resolver_year={h_b1["resolver_year"]}  band_t_from={h_b1["band_t_from"]}  band_t_to={h_b1["band_t_to"]}')

lmr_row = next(r for r in payload['rows']
               if r['variable'] == 'lmr_pdsi' and r.get('year') == 1000)
h_lmr = lmr_row['detail']['distribution']
print()
print('LMR histogram stamp (lmr_pdsi, year=1000):')
print(f'  resolver_year={h_lmr["resolver_year"]}  band_t_from={h_lmr["band_t_from"]}  band_t_to={h_lmr["band_t_to"]}')

print()
print('Summary:')
print('  Band T span (from_year/to_year): payload["temporal"] — top-level ✓')
print('  resolver_year: NOT a top-level key — only in histogram stamps.')
print('  To display "Boundary year: 1000 CE", UI must read a histogram stamp')
print('  or the /api/area route must add it to the neighborhood block.')

In [ ]:
# Cell 9 — lean vs detail delta (checklist item 5)
#
# The ONLY structural difference between lean and detail is the 'detail' key
# on each row. Show exactly what each method's detail sub-dict carries.

print('=== LEAN ROW (first row, aridity) ===')
lean_row = p['s2_lean']['rows'][0]
print(f'  keys: {sorted(lean_row.keys())}')
print(f'  detail field: {lean_row.get("detail")!r}')

print()
print('=== DETAIL SUB-DICT CONTENTS BY METHOD ===')
seen = set()
for r in p['s2_detail']['rows'] + p['s3_detail']['rows']:
    m = r['method']
    if m in seen:
        continue
    d = r.get('detail')
    if d is None:
        print(f'  {m:<30}  detail: null')
    else:
        keys = sorted(d.keys())
        has_hist = 'distribution' in d and d['distribution'] is not None
        print(f'  {m:<30}  detail keys: {keys}  histogram: {has_hist}')
    seen.add(m)

print()
print('Note: top-level "distribution" field exists on every row but is always null.')
print('  lean_row["distribution"]:', lean_row.get('distribution'))
print('The histogram lives in row["detail"]["distribution"], not row["distribution"].')

In [ ]:
# Cell 10 — detail sub-dict deep-dive per method
#
# Show a representative row for each of the 6 base method types
# so we know exactly what the leaf-renderer for each will work with.

rows = p['s2_detail']['rows']

for method in ['area_weighted', 'dominant_basin', 'class_mixture',
               'flag_fraction', 'distribution_only', 'extreme']:
    r = next((x for x in rows if x['method'] == method), None)
    if r is None:
        continue
    print(f'--- {method} | variable: {r["variable"]} ---')
    print(f'  score={r["representative_score"]}  raw={r["representative_raw"]}  '
          f'coherence={r.get("coherence")}  modality={r.get("modality")}')
    d = r.get('detail') or {}
    for k, v in d.items():
        if k == 'distribution' and v is not None:
            print(f'  detail.{k}: {{histogram object, {len(v["bins"])-1} bars}}')
        elif k == 'mixture' and isinstance(v, list):
            print(f'  detail.{k}: [{len(v)} classes] first={v[0]}')
        else:
            print(f'  detail.{k}: {v}')
    print()

In [ ]:
# Cell 11 — basin-ring payload structure
#
# The structurally distinct scope. No top-level rows[] — instead
# {center: <single_basin_payload>, ring: [<member>, ...]}.
# Show the ring member structure and confirm bearing/shared_km fields.

ring = p['s4_lean']

print('Top-level keys:', sorted(ring.keys()))
print(f'lat={ring["lat"]}  lon={ring["lon"]}  level={ring["level"]}')
print(f'center hybas_id: {ring["center"]["neighborhood"]["hybas_id"]}')
print(f'ring members: {len(ring["ring"])}')
print()
print('Ring member metadata (sorted by border_bearing):')
print(f'  {"hybas_id":>12}  {"sub_area_km2":>13}  {"shared_km":>10}  {"border_bearing":>15}  {"centroid_bearing":>17}')
for m in sorted(ring['ring'], key=lambda m: m['border_bearing']):
    print(f'  {m["hybas_id"]:>12}  {m["sub_area_km2"]:>13,.0f}  '
          f'{m["shared_km"]:>10.1f}  {m["border_bearing"]:>15.1f}  {m["centroid_bearing"]:>17.1f}')

print()
print('Each member has a full single_basin_signature payload in member["signature"].')
m0 = ring['ring'][0]
print(f'member 0 signature keys: {sorted(m0["signature"].keys())}')
print(f'member 0 signature rows: {len(m0["signature"]["rows"])}')

In [ ]:
# Cell 12 — 4 Corners / Santa Fe polygon (S5)
#
# The arbitrary-polygon scope. Routes identically to polity minus the name lookup.
# Anasazi / ancestral Puebloan territory: Colorado Plateau, upper Rio Grande.
# Check neighborhood, shortfall, marginal_exposure, and a few key variables.

s5 = p['s5_lean']

print('=== 4 CORNERS / SANTA FE — POLYGON SCOPE ===')
print(f'  bbox: lon -110 to -105.5, lat 35 to 38')
print(f'  neighborhood: {s5["neighborhood"]}')
print(f'  shortfall:    {s5["shortfall"]}')
print()

# Compare to N Song marginal_exposure
s3 = p['s3_lean']
print('Marginal exposure comparison:')
print(f'  N Song (376 basins):      lt_50pct={s3["neighborhood"]["marginal_exposure"]["lt_50pct"]:.4f}  '
      f'lt_20pct={s3["neighborhood"]["marginal_exposure"]["lt_20pct"]:.4f}')
print(f'  4 Corners (28 basins):    lt_50pct={s5["neighborhood"]["marginal_exposure"]["lt_50pct"]:.4f}  '
      f'lt_20pct={s5["neighborhood"]["marginal_exposure"]["lt_20pct"]:.4f}')
print()

# Key environmental variables
print('Selected variables (4 Corners):')
for var in ['aridity', 'run_mm_cyr', 'pre_mm_spr', 'tmp_dc_cmx', 'dis_m3_pyr',
            'snw_pc_cyr', 'ele_mt_sav', 'bio_12']:
    r = next((x for x in s5['rows'] if x['variable'] == var), None)
    if r:
        print(f'  {var:<18}  score={str(r["representative_score"]):>7}  '
              f'coherence={str(r.get("coherence")):>14}  status={r["status"]}')

In [ ]:
# Cell 13 — missing fields survey (checklist item 6)
#
# What the UI design wants that no row currently carries.
# Each gap becomes either a route-layer addition, an engine addition,
# or a surface-computed value.

payload = p['s3_detail']

print('=== GAP SURVEY ===')
print()

# M1: resolver_year at top level
has_top_level = 'resolver_year' in payload
in_nbhd = 'resolver_year' in payload['neighborhood']
# find it in histogram stamp
sample_hist_stamp = next(
    (r['detail']['distribution'] for r in payload['rows']
     if r.get('detail') and r['detail'].get('distribution')),
    {}
)
in_stamp = 'resolver_year' in sample_hist_stamp
print(f'M1 resolver_year:')
print(f'  payload top-level: {has_top_level}')
print(f'  neighborhood block: {in_nbhd}')
print(f'  histogram stamp: {in_stamp}  (value: {sample_hist_stamp.get("resolver_year")})')
print(f'  → gap: not surfaced at a readable top-level location')
print()

# M2: polity name/period
print('M2 polity name + period:')
print(f'  neighborhood keys: {sorted(payload["neighborhood"].keys())}')
print(f'  → gap: polity name and fromyear/toyear not in payload (route-layer addition needed)')
print()

# M3: LMR scalar in lean
lmr_lean = next(r for r in p['s3_lean']['rows'] if r['variable'] == 'lmr_pdsi')
print(f'M3 LMR scalar in lean:')
print(f'  lmr_pdsi lean row — score={lmr_lean["representative_score"]}  detail={lmr_lean.get("detail")}')
print(f'  → gap: no scalar per year in lean; time-series chart requires detail mode')
print()

# M4: B5 distribution_only — no histogram
b5 = next(r for r in p['s2_detail']['rows'] if r['method'] == 'distribution_only')
print(f'M4 B5 distribution_only ({b5["variable"]}) in detail:')
print(f'  detail keys: {sorted(b5.get("detail", {}).keys())}')
print(f'  has histogram: {b5.get("detail", {}).get("distribution") is not None}')
print(f'  → note: range-bar (p10–p90) is the right widget here, not a histogram')
print()

# M5: marginal_exposure on non-polygon scopes
print('M5 marginal_exposure across scopes:')
for label, key in [('S1 single-basin','s1_lean'),('S2 buffer','s2_lean'),
                   ('S3 polity','s3_lean'),('S5 4-corners','s5_lean')]:
    me = p[key]['neighborhood'].get('marginal_exposure', 'ABSENT')
    print(f'  {label}: {me}')
print(f'  → note: conditionally present; UI must handle absence gracefully')